In [ ]:
# --- make the transformer_pipeline package importable from experiments/ ---
import os, sys
_PKG = os.path.abspath(os.path.join(os.getcwd(), "..", "transformer_pipeline"))
if _PKG not in sys.path:
    sys.path.insert(0, _PKG)

# K-fold cross-validation - CLR + scale (the chosen config)

**Why.** Every number so far came from ONE 80/20 split (`val_size=0.20`,
`random_state=42`). With ~209 val samples and run-to-run noise of ~0.03 on
per-feature r, a single split gives a point estimate we cannot put error bars on
- and the transformer's edge over the ridge ceiling is already only +0.02 for
Model A. K-fold replaces that single point with `n_folds` rotations so we report
**mean +/- std** and can see whether the edge survives resampling.

**What this does.** Uses `StratifiedKFoldSplitter` (same composite stratification
label `CENTER_C x PATGROUPFINAL_C`, same rare-stratum merging as the single
split) to cut the COMPLETE samples into `n_folds` folds. Each fold rotates a
different ~1/K slice into validation; the rest is train. For every fold we train
**both** models on the winning config (CLR microbiome + `scale` tokenizer - the
current defaults) and record per-feature r and val_mse at the selected
checkpoint plus the peak over the run.

> FAIR ridge ceilings (per-feature r, same space as the model):
> **A CLR 0.15 | B CLR 0.21**. The question k-fold answers: is the
> transformer's mean clearly above the ceiling, and is the std small enough that
> the gap is real rather than split luck?

Cost note: this is `n_folds x 2` trainings (default 5 x 2 = 10). Trim
`EPOCHS` or `cfg.data.n_folds` if too slow. Run from the repo root: Cell > Run
All.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import numpy as np
import pandas as pd
from pipeline.config import Config
from pipeline.data import OmicsData, StratifiedKFoldSplitter
from pipeline.pipeline import ImputationPipeline

N_FOLDS   = 5          # cfg.data.n_folds
EPOCHS    = 60
PERMS     = 99
TOKENIZER = "scale"    # the winning fusion (default)
CLR       = True       # CLR microbiome (default)

# Build the K folds ONCE (shared by both models, every config).
base = Config()
base.data.n_folds = N_FOLDS
_data = OmicsData.load(base.data)
FOLDS = StratifiedKFoldSplitter(base.data).split(_data)
print(f"{len(FOLDS)} folds | "
      + " ".join(f"f{i}:tr{len(f.train)}/va{len(f.val)}" for i,f in enumerate(FOLDS)))

In [ ]:
def selected(trainer):
    e = trainer.history.best_epoch
    rec = [r for r in trainer.history.records if r.epoch == e][0]
    return rec.per_feature_r, rec.val_mse, (rec.mantel.statistic if rec.mantel else None)

def peak_r(trainer):
    vals = [r.per_feature_r for r in trainer.history.records
            if r.per_feature_r is not None]
    return max(vals) if vals else float("nan")

def run_fold(split):
    """Train A and B on one fold; return per-model dicts."""
    cfg = Config()
    cfg.data.microbiome_clr = CLR
    cfg.model_a.tokenizer = TOKENIZER
    cfg.model_b.tokenizer = TOKENIZER
    cfg.train.max_epochs = EPOCHS
    cfg.train.patience   = EPOCHS      # no early stop -> full curve
    cfg.eval.mantel_permutations = PERMS
    cfg.embedding.unfreeze_random_metabolites = True

    pipe = ImputationPipeline(cfg)
    pipe.prepare_data(split=split)     # <-- inject this fold
    ta = pipe.train_model_a()
    tb = pipe.train_model_b()
    out = {}
    for name, tr in [("A", ta), ("B", tb)]:
        sr, smse, smant = selected(tr)
        out[name] = {"sel_r": sr, "peak_r": peak_r(tr),
                     "sel_val_mse": smse, "sel_mantel": smant}
    return out

fold_results = []
for i, split in enumerate(FOLDS):
    print(f"\n========== FOLD {i+1}/{len(FOLDS)} ==========")
    fold_results.append(run_fold(split))

### Summary: mean +/- std across folds vs the ridge ceilings

In [ ]:
CEIL = {"A": 0.15, "B": 0.21}
NAME = {"A": "A micro->metab", "B": "B metab->micro"}

rows = []
for m in ("A", "B"):
    sel  = np.array([fr[m]["sel_r"]       for fr in fold_results], dtype=float)
    peak = np.array([fr[m]["peak_r"]      for fr in fold_results], dtype=float)
    vmse = np.array([fr[m]["sel_val_mse"] for fr in fold_results], dtype=float)
    rows.append({
        "model": NAME[m],
        "ridge_ceiling":  CEIL[m],
        "sel_r_mean":   round(float(np.nanmean(sel)),  4),
        "sel_r_std":    round(float(np.nanstd(sel)),   4),
        "peak_r_mean":  round(float(np.nanmean(peak)), 4),
        "peak_r_std":   round(float(np.nanstd(peak)),  4),
        "val_mse_mean": round(float(np.nanmean(vmse)), 4),
        "val_mse_std":  round(float(np.nanstd(vmse)),  4),
        "beats_ceiling_by(peak_mean)": round(float(np.nanmean(peak) - CEIL[m]), 4),
    })

cv = pd.DataFrame(rows)
print(f"K = {len(fold_results)} folds   tokenizer={TOKENIZER}   CLR={CLR}")
print("FAIR ridge ceilings (per-feature r):  A CLR 0.15 | B CLR 0.21\n")
print(cv.to_string(index=False))

print("\nPer-fold peak_r:")
for m in ("A", "B"):
    pf = [round(fr[m]["peak_r"], 3) for fr in fold_results]
    print(f"  {NAME[m]}: {pf}")

### Per-fold spread

Each dot is one fold's **peak** per-feature r; the bar is the mean, the line is
the ridge ceiling. If the dots straddle the ceiling line, the transformer's edge
is within split noise; if they sit clearly above it, the edge is real.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
xs = {"A": 0, "B": 1}
for m in ("A", "B"):
    peak = [fr[m]["peak_r"] for fr in fold_results]
    x = xs[m]
    ax.bar(x, np.nanmean(peak), width=0.5, alpha=0.35,
           color="C0" if m == "A" else "C1", zorder=1)
    ax.scatter([x]*len(peak), peak, color="k", zorder=3, s=40)
    ax.hlines(CEIL[m], x-0.3, x+0.3, color="red", lw=2, zorder=2,
              label="ridge ceiling" if m == "A" else None)
    ax.text(x, CEIL[m]+0.003, f"ceiling {CEIL[m]}", ha="center",
            color="red", fontsize=9)
ax.set_xticks([0, 1])
ax.set_xticklabels([NAME["A"], NAME["B"]])
ax.set_ylabel("peak per-feature r")
ax.set_title(f"K-fold ({len(fold_results)} folds) peak r per fold vs ridge ceiling\n"
             f"tokenizer={TOKENIZER}, CLR={CLR}")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### How to read this

- **Mean vs ceiling.** Compare `peak_r_mean` to `ridge_ceiling`. For Model A the
  bar is 0.15; a mean only a hair above it (and a std that crosses it) means the
  transformer is NOT reliably beating linear for A. For Model B the bar is 0.21;
  watch whether the mean is still short of it (the embedding ablation suggested
  B's gap is not an embedding problem).
- **`sel_r` vs `peak_r`.** `sel_r` is what you actually ship (the checkpoint
  selected on val_mse); `peak_r` is the optimistic best-epoch r. A large
  sel<<peak gap across folds means selection is leaving signal on the table /
  the run is overfitting after the selected epoch.
- **std is the point.** A small std with mean above ceiling = a real,
  reproducible edge. A std of ~0.03+ that straddles the ceiling = the single
  split was just luck; report the range honestly.

To change config, edit `TOKENIZER` / `CLR` / `N_FOLDS` in the setup cell. To
sweep tokenizers across folds, wrap `run_fold` in a `for tok in [...]` loop the
way Stage 2+3 does.